# Variable correlations

[Data source](http://archive.ics.uci.edu/ml/datasets/Superconductivty+Data#)

## Imports

In [1]:
from pathlib import Path
import numpy as np
import numpy.polynomial as nppoly
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import scipy.optimize as spopt
import sklearn.preprocessing as skpre
import sklearn.linear_model as sklin
import sklearn.metrics as skmet

In [2]:
plt.rcParams.update({
    #"font.family": "serif",
    #"font.serif": [],                    # use latex default serif font
    #"font.sans-serif": [],
    #"font.size": 20,
    "figure.autolayout": True,
})

from IPython.core import display
from IPython.core.display import HTML
HTML("<style>.container { width:100% !important; }</style>")

## Constants

In [3]:
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "train.csv"
ELEM_FILE = DATA_DIR / "unique_m.csv"

FIGSIZE_GRAPH = [10, 7]

TARGET = "critical_temp"

## Help functions

In [4]:
def print_labels(labels):
    for label in labels:
        print("\"" + label + "\",")

def power_func(x, a, nu, b):
    return a*np.power(x, nu) + b
        
def make_pair_plot(data_obj, plot_vars):
    sns.pairplot(data = data_obj,
                 vars = plot_vars)

    plt.suptitle("PairPlot of Data",fontsize=18)

    plt.tight_layout()

def make_correlation_matrix(data_obj, labels, thres):
    plt.figure(figsize = (16, 12))
    sns.heatmap(data = data_obj[labels].corr(),
                vmin = -thres,
                vmax = thres,
                annot=True,
                cmap="bone",
                linewidths=0.1,
                fmt=".2f",
                linecolor="gray")
    
    plt.title("Correlation Matrix")
    
    plt.tight_layout()

def plot_vars_target(data_obj, plot_vars, target):
    for var in plot_vars:
        fig = plt.figure(figsize = FIGSIZE_GRAPH)
        ax = fig.add_subplot()

        var_data = data_obj.get(var)
        order_keys = np.argsort(var_data)

        ax.scatter(var_data[order_keys], data_obj.get(target)[order_keys], s = 1)
        ax.set_xlabel(var)
        ax.set_ylabel(target)

## Data preparation

In [5]:
data_obj = pd.read_csv(DATA_FILE) # Data with general features and critical temperature
elem_obj = pd.read_csv(ELEM_FILE) # Data with chemical composition

## Correlations with target

In [6]:
target_values = data_obj.get(TARGET)

corrs_for_column: dict[str, float] = {}

for column in data_obj.columns.values:
    if column != TARGET:
        corrs_for_column[column] = target_values.corr(data_obj.get(column))

sorted_corrs_for_column = {col: corr for col, corr in sorted(corrs_for_column.items(), key=lambda item: np.abs(item[1]), reverse=True)}

for col, corr in sorted_corrs_for_column.items():
    print(f"- {col}: {corr}")

- wtd_std_ThermalConductivity: 0.7212710791834754
- range_ThermalConductivity: 0.6876539119498933
- range_atomic_radius: 0.6537590446423304
- std_ThermalConductivity: 0.653631981546976
- wtd_mean_Valence: -0.6324010170934355
- wtd_entropy_atomic_mass: 0.626930401693083
- wtd_gmean_Valence: -0.6156532964179642
- wtd_entropy_atomic_radius: 0.603493983333192
- number_of_elements: 0.6010685709560267
- range_fie: 0.6007903800812856
- mean_Valence: -0.6000848649246685
- wtd_std_atomic_radius: 0.5991986591462265
- entropy_Valence: 0.5985909069318409
- wtd_entropy_Valence: 0.5896637026115057
- wtd_std_fie: 0.5820132554394737
- gmean_Valence: -0.5730680556134586
- entropy_fie: 0.5678169384811448
- wtd_entropy_FusionHeat: 0.5632442680994024
- std_atomic_radius: 0.5596285739605901
- entropy_atomic_radius: 0.5589374384154047
- entropy_FusionHeat: 0.5527087051788929
- entropy_atomic_mass: 0.5436194092285056
- std_fie: 0.5418038101921668
- gmean_Density: -0.5416844072618782
- wtd_gmean_Density: -0.5

Initial candidates for variables taken at threshold of >0.5 absolute correlation with target:

- `wtd_std_ThermalConductivity`
- `range_ThermalConductivity`
- `range_atomic_radius`
- `std_ThermalConductivity`
- `wtd_mean_Valence`
- `wtd_entropy_atomic_mass`
- `wtd_gmean_Valence`
- `wtd_entropy_atomic_radius`
- `number_of_elements`
- `range_fie`
- `mean_Valence`
- `wtd_std_atomic_radius`
- `entropy_Valence`
- `wtd_entropy_Valence`
- `wtd_std_fie`
- `gmean_Valence`
- `entropy_fie`
- `wtd_entropy_FusionHeat`
- `std_atomic_radius`
- `entropy_atomic_radius`
- `entropy_FusionHeat`
- `entropy_atomic_mass`
- `std_fie`
- `gmean_Density`
- `wtd_gmean_Density`

## Feature redundancy and correlations

In [31]:
feature_columns = [column for column in data_obj.columns.values if column != TARGET]

corrs = np.corrcoef(data_obj.get(feature_columns), rowvar=False)

correlations_above_thres = np.argwhere(corrs > 0.75)

In [ ]:
correlate_indices = {}

for i, j in correlations_above_thres:
    if i not in correlate_indices.keys():
        correlate_indices[int(i)] = []
    if i < j:
        correlate_indices[int(i)].append(int(j))

In [19]:
def correlates_with_all(index: int, existing_cluster: list[int], correlate_indices: dict[int, list[int]]) -> bool:
    return all([index in correlate_indices[member] for member in existing_cluster])

In [29]:
clusters = []

for i in correlate_indices.keys():
    if any(i in cluster for cluster in clusters):
        continue
    
    cluster = [i]
    i_correlates = correlate_indices[i]

    for correlate in i_correlates:
        if correlates_with_all(correlate, cluster, correlate_indices):
            cluster.append(correlate)

    if len(cluster) > 1:
        clusters.append(cluster)

In [33]:
for cluster in clusters:
    print("Correlations: " + ", ".join([feature_columns[i] for i in cluster]))

Correlations: number_of_elements, entropy_atomic_mass, wtd_entropy_atomic_mass, entropy_fie, entropy_atomic_radius, wtd_entropy_atomic_radius, entropy_Density, wtd_entropy_Density, entropy_FusionHeat, wtd_entropy_FusionHeat, entropy_Valence, wtd_entropy_Valence
Correlations: mean_atomic_mass, wtd_mean_atomic_mass, gmean_atomic_mass
Correlations: wtd_gmean_atomic_mass, wtd_mean_Density, gmean_Density, wtd_gmean_Density
Correlations: range_atomic_mass, std_atomic_mass, wtd_std_atomic_mass
Correlations: wtd_range_atomic_mass, wtd_range_Density
Correlations: mean_fie, gmean_fie
Correlations: wtd_mean_fie, wtd_gmean_fie, wtd_std_fie
Correlations: wtd_entropy_fie, wtd_entropy_atomic_radius, wtd_entropy_Density, wtd_entropy_Valence
Correlations: range_fie, std_fie, wtd_std_fie, range_atomic_radius, std_atomic_radius, wtd_std_atomic_radius
Correlations: mean_atomic_radius, gmean_atomic_radius
Correlations: wtd_mean_atomic_radius, wtd_gmean_atomic_radius
Correlations: mean_Density, wtd_mean_Den

Some features are redundant versions of each other. From among each such set, we select only the single feature that correlates the strongest with the target. By manual and code-assisted search these are found to be:
* `"mean_atomic_mass", "wtd_mean_atomic_mass", "gmean_atomic_mass", "wtd_gmean_atomic_mass"`. `"wtd_gmean_atomic_mass"` correlates the strongest with critical temperature.
* `"entropy_atomic_mass", "wtd_entropy_atomic_mass"`. `"wtd_entropy_atomic_mass"` correlates the strongest with critical temperature.
* `"range_atomic_mass", "std_atomic_mass", "wtd_std_atomic_mass"`. `"range_atomic_mass"` correlates the strongest with critical temperature.
* `"mean_fie", "gmean_fie"`. `"mean_fie"` correlates the strongest with critical temperature.
* `"wtd_mean_fie", "wtd_gmean_fie"`. `"wtd_mean_fie"` correlates the strongest with critical temperature.
* `"entropy_fie", "wtd_entropy_fie"`. `"entropy_fie"` correlates the strongest with critical temperature.
* `"range_fie", "std_fie", "wtd_std_fie",`. `"range_fie"` correlates the strongest with critical temperature.
* `"mean_atomic_radius", "gmean_atomic_radius"`. `"gmean_atomic_radius"` correlates the strongest with critical temperature.
* `"wtd_mean_atomic_radius", "wtd_gmean_atomic_radius"`. `"wtd_gmean_atomic_radius"` correlates the strongest with critical temperature.
* `"entropy_atomic_radius", "wtd_entropy_atomic_radius"`. `"wtd_entropy_atomic_radius"` correlates the strongest with critical temperature.
* `"range_atomic_radius", "std_atomic_radius", "wtd_std_atomic_radius"`. `"range_atomic_radius"` correlates the strongest with critical temperature.
* `"mean_Density", "wtd_mean_Density", "gmean_Density", "wtd_gmean_Density"`. `"gmean_Density", "wtd_gmean_Density"` correlate the strongest with critical temperature.
* `"entropy_Density", "wtd_entropy_Density"`. `"entropy_Density"` correlates the strongest with critical temperature.
* `"range_Density", "std_Density", "wtd_std_Density"`. `"range_Density"` correlates the strongest with critical temperature.
* `"mean_ElectronAffinity", "gmean_ElectronAffinity",`. `"mean_ElectronAffinity"` correlates the strongest with critical temperature.
* `"wtd_mean_ElectronAffinity", "wtd_gmean_ElectronAffinity"`. `"wtd_mean_ElectronAffinity"` correlates the strongest with critical temperature.
* `"range_ElectronAffinity", "std_ElectronAffinity", "wtd_std_ElectronAffinity"`. `"wtd_std_ElectronAffinity",` correlates the strongest with critical temperature.
* `"mean_FusionHeat", "wtd_mean_FusionHeat", "gmean_FusionHeat", "wtd_gmean_FusionHeat"`. `"gmean_FusionHeat",
"wtd_gmean_FusionHeat"` correlate the strongest with critical temperature.
* `"entropy_FusionHeat", "wtd_entropy_FusionHeat"`. `"wtd_entropy_FusionHeat"` correlates the strongest with critical temperature.
* `"range_FusionHeat", "std_FusionHeat", "wtd_std_FusionHeat"`. `"std_FusionHeat", "wtd_std_FusionHeat"` correlate equally with critical temperature.
* `"gmean_ThermalConductivity", "wtd_gmean_ThermalConductivity"`. `"gmean_ThermalConductivity"` correlates the strongest with critical temperature.
* `"entropy_ThermalConductivity", "wtd_entropy_ThermalConductivity"`. `"wtd_entropy_ThermalConductivity"` correlates the strongest with critical temperature.
* `"range_ThermalConductivity", "std_ThermalConductivity", "wtd_std_ThermalConductivity"`. `"wtd_std_ThermalConductivity"` correlates the strongest with critical temperature.
* `"mean_Valence", "wtd_mean_Valence", "gmean_Valence", "wtd_gmean_Valence"`. `"wtd_mean_Valence"` correlates the strongest with critical temperature.
* `"entropy_Valence", "wtd_entropy_Valence"`. `"entropy_Valence"` correlates the strongest with critical temperature.
* `"range_Valence", "std_Valence", "wtd_std_Valence",`. `"wtd_std_Valence"` correlates the strongest with critical temperature.

## Redundant feature removal

In [47]:
feature_columns = [column for column in data_obj.columns.values if column != TARGET]

redundant_features = ["mean_atomic_mass", "wtd_mean_atomic_mass", "gmean_atomic_mass", 
                    "entropy_atomic_mass", 
                    "std_atomic_mass", "wtd_std_atomic_mass",
                    "gmean_fie",
                    "wtd_gmean_fie",
                    "wtd_entropy_fie",
                    "std_fie", "wtd_std_fie",
                    "mean_atomic_radius", 
                    "wtd_mean_atomic_radius", 
                    "entropy_atomic_radius", 
                    "std_atomic_radius", "wtd_std_atomic_radius", 
                    "mean_Density", "wtd_mean_Density", "gmean_Density",
                    "wtd_entropy_Density", 
                    "std_Density", "wtd_std_Density", 
                    "gmean_ElectronAffinity",
                    "wtd_gmean_ElectronAffinity", 
                    "range_ElectronAffinity", "std_ElectronAffinity", 
                    "mean_FusionHeat", "wtd_mean_FusionHeat", "gmean_FusionHeat", 
                    "entropy_FusionHeat", 
                    "range_FusionHeat", "std_FusionHeat", 
                    "wtd_gmean_ThermalConductivity", 
                    "entropy_ThermalConductivity", 
                    "range_ThermalConductivity", "std_ThermalConductivity", 
                    "mean_Valence", "gmean_Valence", "wtd_gmean_Valence",
                    "wtd_entropy_Valence",
                    "range_Valence", "std_Valence", 
]

unique_features = [column for column in feature_columns if column not in redundant_features]

## Correlation threshold imposal

In [74]:
correlated_features = []

for feature in unique_features:
    if abs(corrs_for_column[feature]) > 0.5:
        correlated_features.append(feature)

correlated_features = sorted(correlated_features, key=lambda feat: abs(corrs_for_column[feat]), reverse=True)

The following properties correlate strongly with critical temperature:
* `wtd_std_ThermalConductivity`
* `range_atomic_radius`
* `wtd_mean_Valence`
* `wtd_entropy_atomic_mass`
* `wtd_entropy_atomic_radius`
* `number_of_elements`
* `range_fie`
* `entropy_Valence`
* `entropy_fie`
* `wtd_entropy_FusionHeat`
* `wtd_gmean_Density`